<a href="https://colab.research.google.com/github/Areej973/Building-a-Simple-RAG-System-from-Scratch/blob/main/Building_a_Simple_RAG_System_from_Scratch11.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Building a Simple RAG System from Scratch

In [1]:
!pip install faiss-cpu sentence-transformers transformers datasets==2.14.5

INFO: pip is looking at multiple versions of transformers to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of transformers to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.9 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of multiprocess to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 519.6/519.6 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 31.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 53.8 MB/

In [2]:
from sentence_transformers import SentenceTransformer
from transformers import pipeline
import faiss
import numpy as np
import pandas as pd

**SAMPLE DOCUMENTS**

In [3]:
documents = ["The capital of Canada is Ottawa.", "Python is a popular programming language for AI and machine learning.", "The Great Wall of China is visible from space is a myth.", "FAISS is a library for efficient similarity search and clustering of dense vectors.", "Transformers models are widely used for natural language processing tasks."]

**EMBEDDING MODEL**

In [4]:
embedding_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
doc_embeddings = embedding_model.encode(documents, convert_to_tensor=True, normalize_embeddings=True)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

 **CREATE FAISS INDEX**

In [5]:
dimension = doc_embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(doc_embeddings)

print(f"FAISS index contains {index.ntotal} documents.")

FAISS index contains 5 documents.


**RETRIEVAL FUNCTION**

In [6]:
def retrieve(query, top_k=2):
    query_emb = embedding_model.encode([query], normalize_embeddings=True)
    distances, indices = index.search(query_emb, top_k)
    retrieved_docs = [documents[i] for i in indices[0]]
    return retrieved_docs

**GENERATION MODEL (QA or Summarization)**

In [7]:
generator = pipeline("text2text-generation", model="google/flan-t5-base")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Device set to use cpu


**RAG PIPELINE FUNCTION**

In [8]:
def rag_pipeline(query):
    retrieved_docs = retrieve(query)
    context = " ".join(retrieved_docs)
    prompt = f"Context: {context}\nQuestion: {query}\nAnswer:"
    response = generator(prompt, max_length=100, temperature=0.3)[0]['generated_text']

    print("🔍 Retrieved Documents:\n", retrieved_docs)
    print("\n💬 Model Answer:\n", response)

**TEST THE PIPELINE**

In [9]:
rag_pipeline("What is FAISS used for?")
rag_pipeline("Where is the capital of Canada?")

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Both `max_new_tokens` (=256) and `max_length`(=100) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🔍 Retrieved Documents:
 ['FAISS is a library for efficient similarity search and clustering of dense vectors.', 'Python is a popular programming language for AI and machine learning.']

💬 Model Answer:
 efficient similarity search and clustering of dense vectors


Both `max_new_tokens` (=256) and `max_length`(=100) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🔍 Retrieved Documents:
 ['The capital of Canada is Ottawa.', 'The Great Wall of China is visible from space is a myth.']

💬 Model Answer:
 Ottawa
